In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
from matplotlib import pyplot as plt 
from scipy import stats 

In [2]:
df = pd.read_csv('../insurance_lab/data/processed/insurance_clean.csv')
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1332,50,male,30.970,3,no,northwest,10600.54830
1333,18,female,31.920,0,no,northeast,2205.98080
1334,18,female,36.850,0,no,southeast,1629.83350
1335,21,female,25.800,0,no,southwest,2007.94500


In [3]:
print(f'shape : {df.shape} ')
print(df[['smoker','charges']])

shape : (1337, 7) 
     smoker      charges
0       yes  16884.92400
1        no   1725.55230
2        no   4449.46200
3        no  21984.47061
4        no   3866.85520
...     ...          ...
1332     no  10600.54830
1333     no   2205.98080
1334     no   1629.83350
1335     no   2007.94500
1336    yes  29141.36030

[1337 rows x 2 columns]


In [4]:
df['smoker'].value_counts()

smoker
no     1063
yes     274
Name: count, dtype: int64

In [5]:
smoker_charges = df[df['smoker'] == 'yes']['charges']
non_smoker_charges = df[df['smoker'] == 'no']['charges']
# skewnss
smoker_skew = smoker_charges.skew()
non_smoker_skew = non_smoker_charges.skew()
# levene test if var between the sampls are equal 
t_levene,p_value = stats.levene(smoker_charges,non_smoker_charges)

print(f'{'kind':<5} | {'value':<10}')
print('-'*50)
print(f'{'somker_skew':<10} | {smoker_skew :<10.4f}')
print(f'{'non_somker_skew':<10} | {non_smoker_skew:<10.4f}')
print(f'{'p_value':<10} | {p_value:<10}')
print(f'{'t_test':<10} | {t_levene:<10.4f}')


kind  | value     
--------------------------------------------------
somker_skew | 0.1282    
non_somker_skew | 1.5384    
p_value    | 1.6701175651271788e-66
t_test     | 332.4714  


In [6]:
# welch test 
t_stat , p_val_welch = stats.ttest_ind(smoker_charges,non_smoker_charges,equal_var=False)
# diffrent mean
mean_diff = smoker_charges.mean() - non_smoker_charges.mean()
# stander error
se_diff = np.sqrt((smoker_charges.var(ddof=1)/len(smoker_charges)) + (non_smoker_charges.var(ddof=1)/len(non_smoker_charges)))

v1 , v2 = smoker_charges.var(ddof=1) , non_smoker_charges.var(ddof=1)
n1 ,n2 = len(smoker_charges) , len(non_smoker_charges)
df_welch = ((v1/n1 + v2/n2)**2)/(((v1/n1)**2/(n1-1)) + ((v2/n2)**2/(n2-1)))
t_crit = stats.t.ppf(0.975,df_welch)
ic = [mean_diff - t_crit * se_diff , mean_diff + t_crit * se_diff]

u = stats.mannwhitneyu(smoker_charges,non_smoker_charges)

print(f't_stat : {t_stat : .5f} | p_value_welch : {p_val_welch:.4f}')
print(f'SE : {se_diff}')
print(f'IC : from : {ic[0]:.4f} | to : {ic[1]:.4f}')
print(f'mann whitney U : t test : {u[0]:.4f} | p_value : {u[1]:.4f}')

t_stat :  32.74231 | p_value_welch : 0.0000
SE : 721.0722674876497
IC : from : 22190.7901 | to : 25028.3530
mann whitney U : t test : 283859.0000 | p_value : 0.0000


In [8]:
s_pooled = np.sqrt(((n1 - 1) * v1  + (n2 -1) * v2) / (n1 + n2 - 2))

cohen_d = abs(mean_diff) / s_pooled

print(f'pooled standard deviation : {s_pooled:.2f}')
print(f'cohen d : {cohen_d :.4f} ')

pooled standard deviation : 7470.70
cohen d : 3.1603 
